# Notebook 2 — Preprocessing & Feature Engineering
**AAI-590 Capstone | Validex Growth Investors**

### Purpose
Loads data saved by `01_data_pull.ipynb` and builds the **modeling-ready dataset**.  
No API calls are made here 

### Pipeline
```
Section 0  → Config & Load Raw Data
Section 1  → Price-Based Feature Engineering
Section 2  → Fundamental & Valuation Feature Engineering
Section 3  → Align Quarterly Fundamentals to Daily (Point-in-Time Safe)
Section 4  → Options Processing & Bucket Classification
Section 5  → Target Label Construction (Sharpe-Based)
Section 6  → Final Dataset Assembly & Export
```

### Input
```
data/processed/
├── daily_adjusted/ALL_daily_adjusted.parquet
├── fundamentals/ALL_income_statement.parquet
├── fundamentals/ALL_balance_sheet.parquet
├── fundamentals/ALL_cash_flow.parquet
├── fundamentals/ALL_overview.csv
└── options/ALL_options.parquet
```

### Output
```
data/final/
├── modeling_dataset.parquet   ← full labeled dataset (if options available)
├── modeling_dataset.csv
├── features_only.parquet      ← price + fundamentals (always produced)
└── features_only.csv
```


## Section 0 — Config & Load Raw Data

In [8]:
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.4f}".format)

print(" Imports OK")

 Imports OK


In [9]:
#  Settings 
TICKERS    = ["ADMA", "NTRA", "AXON", "SHAK", "AAPL", "MSFT", "NVDA", "AMZN", "GOOG", "META"]
DATE_START = "2015-01-01"
DATE_END   = "2025-12-31"
RISK_FREE  = 0.05   # annualized risk-free rate for Sharpe computation

# ── Paths ─────────────────────────────────────────────────────────────────────
PROC = Path("../data/processed")
FEAT = Path("../data/processed/features")
FINAL= Path("../data/final")
for d in [FEAT, FINAL]:
    d.mkdir(parents=True, exist_ok=True)

# ── All 9 valid bucket labels ─────────────────────────────────────────────────
VALID_BUCKETS = [f"{m}_{d}" for m in ["ATM","OTM5","OTM10"] for d in [30,60,90]]
BUCKET_TO_INT = {b: i for i, b in enumerate(VALID_BUCKETS)}

print(f"Buckets ({len(VALID_BUCKETS)}): {VALID_BUCKETS}")

Buckets (9): ['ATM_30', 'ATM_60', 'ATM_90', 'OTM5_30', 'OTM5_60', 'OTM5_90', 'OTM10_30', 'OTM10_60', 'OTM10_90']


In [10]:
# ── Load Processed Data from Notebook 1 ──────────────────────────────────────
def load_parquet_safe(path: Path, label: str) -> pd.DataFrame:
    if path.exists():
        df = pd.read_parquet(path)
        print(f"   {label}: {df.shape}")
        return df
    else:
        print(f"   {label}: NOT FOUND at {path}  → run 01_data_pull.ipynb first")
        return pd.DataFrame()

print("Loading data...")
daily_raw    = load_parquet_safe(PROC / "daily_adjusted/ALL_daily_adjusted.parquet",    "Daily prices")
income_raw   = load_parquet_safe(PROC / "fundamentals/ALL_income_statement.parquet",    "Income statement")
balance_raw  = load_parquet_safe(PROC / "fundamentals/ALL_balance_sheet.parquet",       "Balance sheet")
cashflow_raw = load_parquet_safe(PROC / "fundamentals/ALL_cash_flow.parquet",           "Cash flow")
options_raw  = load_parquet_safe(PROC / "options/ALL_options.parquet",                  "Options")

# Overview is CSV
overview_path = PROC / "fundamentals/ALL_overview.csv"
overview_raw  = pd.read_csv(overview_path) if overview_path.exists() else pd.DataFrame()
print(f"  { 'ok'if not overview_raw.empty else 'not ok'} Overview: {overview_raw.shape}")

# Ensure date columns are datetime
if not daily_raw.empty:
    daily_raw["date"] = pd.to_datetime(daily_raw["date"])
for df in [income_raw, balance_raw, cashflow_raw]:
    if not df.empty and "fiscalDateEnding" in df.columns:
        df["fiscalDateEnding"] = pd.to_datetime(df["fiscalDateEnding"])

Loading data...
   Daily prices: (27516, 10)
   Income statement: (736, 27)
   Balance sheet: (729, 39)
   Cash flow: (731, 30)
   Options: (1854022, 20)
  ok Overview: (10, 55)



## Section 1 — Price-Based Feature Engineering
Computes returns, momentum, moving averages, realized volatility, drawdown, and volume features.

In [ ]:
def engineer_price_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Input : single-ticker daily DataFrame, sorted by date.
    Output: same DataFrame with price-based features appended.
    """
    df = df.sort_values("date").copy()

    # ── Returns ────────────────────────────────────────────────────────────────
    df["return_1d"]    = df["adj_close"].pct_change()
    df["log_return"]   = np.log(df["adj_close"] / df["adj_close"].shift(1))

    # ── Momentum ───────────────────────────────────────────────────────────────
    df["momentum_1m"]  = df["adj_close"].pct_change(21)    # ~1 month
    df["momentum_3m"]  = df["adj_close"].pct_change(63)    # ~3 months
    df["momentum_6m"]  = df["adj_close"].pct_change(126)   # ~6 months
    df["momentum_12m"] = df["adj_close"].pct_change(252)   # ~12 months

    # ── Moving Averages & Price vs MA ─────────────────────────────────────────
    df["ma_20"]          = df["adj_close"].rolling(20).mean()
    df["ma_50"]          = df["adj_close"].rolling(50).mean()
    df["ma_200"]         = df["adj_close"].rolling(200).mean()
    df["price_vs_ma20"]  = df["adj_close"] / df["ma_20"]  - 1
    df["price_vs_ma50"]  = df["adj_close"] / df["ma_50"]  - 1
    df["price_vs_ma200"] = df["adj_close"] / df["ma_200"] - 1
    df["ma50_vs_ma200"]  = df["ma_50"]     / df["ma_200"] - 1   # golden/death cross proxy

    # ── Realized Volatility (annualized) ──────────────────────────────────────
    df["vol_20d"]  = df["return_1d"].rolling(20).std()  * np.sqrt(252)
    df["vol_60d"]  = df["return_1d"].rolling(60).std()  * np.sqrt(252)
    df["vol_120d"] = df["return_1d"].rolling(120).std() * np.sqrt(252)

    # ── Parkinson High-Low Volatility ─────────────────────────────────────────
    df["parkinson_vol_20d"] = np.sqrt(
        (1 / (4 * np.log(2))) *
        (np.log(df["high"] / df["low"]) ** 2).rolling(20).mean()
    ) * np.sqrt(252)

    # ── Intraday Range ────────────────────────────────────────────────────────
    df["intraday_range"] = (df["high"] - df["low"]) / df["close"]

    # ── Drawdown & Market Regime ───────────────────────────────────────────────
    df["rolling_max"]     = df["adj_close"].cummax()
    df["drawdown"]        = (df["adj_close"] / df["rolling_max"]) - 1
    df["drawdown_regime"] = (df["drawdown"] < -0.20).astype(int)   # 1 = deep drawdown
    df["near_52w_high"]   = df["adj_close"] / df["adj_close"].rolling(252).max()

    # ── Volume Features ───────────────────────────────────────────────────────
    df["volume_ma20"]   = df["volume"].rolling(20).mean()
    df["volume_spike"]  = df["volume"] / df["volume_ma20"]
    df["volume_change"] = df["volume"].pct_change()

    # ── Dividend Yield (proxy) ────────────────────────────────────────────────
    df["dividend_yield"] = df["dividend"] / df["adj_close"]

    return df

print("engineer_price_features defined")

In [ ]:
# ── Apply to All Tickers ──────────────────────────────────────────────────────
price_feat_dfs = []

for sym in tqdm(TICKERS, desc="Price Features"):
    df_sym = daily_raw[daily_raw["symbol"] == sym].copy()
    if df_sym.empty:
        print(f" {sym}: no daily data")
        continue
    df_feat = engineer_price_features(df_sym)
    price_feat_dfs.append(df_feat)

daily_features = pd.concat(price_feat_dfs, ignore_index=True)
daily_features.to_parquet(FEAT / "ALL_price_features.parquet", index=False)

print(f"\n Price features: {daily_features.shape}")
print(f"   New columns: {[c for c in daily_features.columns if c not in daily_raw.columns]}")
daily_features.head(3)

---
## Section 2 — Fundamental & Valuation Feature Engineering
Merges quarterly statements and computes profitability, growth, and balance sheet ratios.

In [ ]:
def build_fundamental_features(income: pd.DataFrame,
                               balance: pd.DataFrame,
                               cashflow: pd.DataFrame) -> pd.DataFrame:
    """
    Merges quarterly income, balance sheet, and cash flow statements.
    Computes derived valuation and profitability metrics.
    Returns one row per (symbol, quarter).
    """
    if income.empty:
        return pd.DataFrame()

    df = income.merge(
        balance,  on=["symbol","fiscalDateEnding"], how="outer", suffixes=("","_bal")
    ).merge(
        cashflow, on=["symbol","fiscalDateEnding"], how="outer", suffixes=("","_cf")
    )
    df = df.sort_values(["symbol","fiscalDateEnding"]).reset_index(drop=True)

    # ── Profitability Margins ──────────────────────────────────────────────────
    def safe_ratio(num, den):
        return df[num] / df[den].replace(0, np.nan) if (num in df.columns and den in df.columns) else np.nan

    df["gross_margin"]  = safe_ratio("grossProfit",      "totalRevenue")
    df["ebitda_margin"] = safe_ratio("ebitda",           "totalRevenue")
    df["net_margin"]    = safe_ratio("netIncome",         "totalRevenue")
    df["ocf_margin"]    = safe_ratio("operatingCashflow", "totalRevenue")

    # ── Revenue & Earnings Growth (YoY = 4 quarters) ──────────────────────────
    growth_records = []
    for sym, grp in df.groupby("symbol"):
        g = grp.copy()
        if "totalRevenue" in g.columns:
            g["revenue_growth_qoq"] = g["totalRevenue"].pct_change(1)
            g["revenue_growth_yoy"] = g["totalRevenue"].pct_change(4)
        if "netIncome" in g.columns:
            g["earnings_growth_yoy"] = g["netIncome"].pct_change(4)
        if "grossProfit" in g.columns:
            g["gross_profit_growth_yoy"] = g["grossProfit"].pct_change(4)
        growth_records.append(g)

    df = pd.concat(growth_records, ignore_index=True)

    # ── Balance Sheet Ratios ───────────────────────────────────────────────────
    df["current_ratio"]  = safe_ratio("totalCurrentAssets",   "totalCurrentLiabilities")
    df["debt_to_equity"] = safe_ratio("totalLiabilities",     "totalShareholderEquity")
    df["asset_turnover"] = safe_ratio("totalRevenue",         "totalAssets")
    df["roe"]            = safe_ratio("netIncome",            "totalShareholderEquity")
    df["roa"]            = safe_ratio("netIncome",            "totalAssets")

    # ── Cash Flow ─────────────────────────────────────────────────────────────
    if all(c in df.columns for c in ["operatingCashflow","capitalExpenditures"]):
        df["fcf"]        = df["operatingCashflow"] - df["capitalExpenditures"].abs()
        df["capex_ratio"]= df["capitalExpenditures"].abs() / df["operatingCashflow"].replace(0, np.nan)

    return df

print(" build_fundamental_features defined")

In [ ]:
# ── Build Quarterly Feature Table ─────────────────────────────────────────────
quarterly_features = build_fundamental_features(income_raw, balance_raw, cashflow_raw)

if not quarterly_features.empty:
    quarterly_features.to_parquet(FEAT / "ALL_fundamental_features.parquet", index=False)
    quarterly_features.to_csv(FEAT / "ALL_fundamental_features.csv", index=False)
    print(f"Quarterly fundamental features: {quarterly_features.shape}")

    display_cols = ["symbol","fiscalDateEnding","gross_margin","net_margin",
                    "revenue_growth_yoy","debt_to_equity","roe","fcf"]
    quarterly_features[[c for c in display_cols if c in quarterly_features.columns]].head(8)
else:
    print(" No fundamental data — skipping.")

## Section 3 — Align Quarterly Fundamentals to Daily (Point-in-Time Safe)
Uses `pd.merge_asof` to forward-fill the most recently published quarterly report onto each trading day — **no look-ahead bias**.

In [ ]:
def align_fundamentals_to_daily(daily_df: pd.DataFrame,
                                 quarterly_df: pd.DataFrame) -> pd.DataFrame:
    """
    For each (symbol, date), attach the most recently available quarterly report
    using a backward asof merge — point-in-time safe (no future data leakage).
    """
    if quarterly_df.empty:
        print("  No quarterly data to align — returning price features only.")
        return daily_df

    daily_df     = daily_df.sort_values(["symbol","date"]).copy()
    quarterly_df = quarterly_df.sort_values(["symbol","fiscalDateEnding"]).copy()

    # Columns to carry over from fundamentals (exclude merge keys & metadata)
    exclude = {"symbol","fiscalDateEnding","reportedCurrency","reportedCurrency_bal","reportedCurrency_cf"}
    fund_cols = [c for c in quarterly_df.columns if c not in exclude]

    all_merged = []
    for sym in daily_df["symbol"].unique():
        d = daily_df[daily_df["symbol"] == sym].copy()
        q = quarterly_df[quarterly_df["symbol"] == sym][["fiscalDateEnding"] + fund_cols].copy()
        q = q.rename(columns={"fiscalDateEnding": "date"})
        q["date"] = pd.to_datetime(q["date"])

        # For each daily row: use the LAST quarterly report on or before that date
        merged = pd.merge_asof(d, q, on="date", direction="backward")
        all_merged.append(merged)

    return pd.concat(all_merged, ignore_index=True)


print(" align_fundamentals_to_daily defined")

In [ ]:
# ── Align & Save ──────────────────────────────────────────────────────────────
daily_with_fundamentals = align_fundamentals_to_daily(daily_features, quarterly_features)
daily_with_fundamentals.to_parquet(FEAT / "ALL_daily_with_fundamentals.parquet", index=False)

print(f" Daily + fundamentals: {daily_with_fundamentals.shape}")

# Quick check: fundamental columns populated?
fund_check_cols = ["gross_margin","net_margin","revenue_growth_yoy","debt_to_equity"]
available = [c for c in fund_check_cols if c in daily_with_fundamentals.columns]
if available:
    print("\nSample fundamental fill rates:")
    print(daily_with_fundamentals[available].notna().mean().round(3))

---
## Section 4 — Options Processing & Bucket Classification
Joins options to spot prices, computes DTE, moneyness, and assigns each call option to one of 9 covered call buckets.

In [ ]:
# ── Bucket Classification Functions ───────────────────────────────────────────
def classify_moneyness(spot: float, strike: float,
                       atm_band: float    = 0.01,
                       otm5_center: float = 0.05, otm5_band: float  = 0.01,
                       otm10_center:float = 0.10, otm10_band: float = 0.01):
    """
    ATM  : strike within ±1% of spot
    OTM5 : strike 4%–6% above spot
    OTM10: strike 9%–11% above spot
    Returns: 'ATM', 'OTM5', 'OTM10', or None
    """
    if pd.isna(spot) or spot <= 0 or pd.isna(strike):
        return None
    m = (strike / spot) - 1.0
    if abs(m) <= atm_band:                                                   return "ATM"
    if (otm5_center  - otm5_band)  <= m <= (otm5_center  + otm5_band):      return "OTM5"
    if (otm10_center - otm10_band) <= m <= (otm10_center + otm10_band):      return "OTM10"
    return None


def classify_dte(dte,
                 dte_30=(25, 35),
                 dte_60=(55, 65),
                 dte_90=(85, 95)):
    """
    30-DTE bucket: 25–35 days
    60-DTE bucket: 55–65 days
    90-DTE bucket: 85–95 days
    Returns: 30, 60, 90, or None
    """
    if pd.isna(dte): return None
    dte = int(dte)
    if dte_30[0] <= dte <= dte_30[1]: return 30
    if dte_60[0] <= dte <= dte_60[1]: return 60
    if dte_90[0] <= dte <= dte_90[1]: return 90
    return None


def make_bucket_label(spot, strike, dte):
    m = classify_moneyness(spot, strike)
    d = classify_dte(dte)
    return f"{m}_{d}" if (m and d) else None

print(" Bucket classification functions defined")
print(f"   Valid buckets: {VALID_BUCKETS}")

In [ ]:
# ── Process Options & Assign Buckets ─────────────────────────────────────────
if not options_raw.empty and "trade_date" in options_raw.columns:

    df_opt = options_raw.copy()
    df_opt["trade_date"] = pd.to_datetime(df_opt["trade_date"])
    df_opt["expiration"] = pd.to_datetime(df_opt["expiration"])

    # Keep CALL options only (covered call = selling calls)
    if "call_put" in df_opt.columns:
        df_opt["call_put"] = df_opt["call_put"].astype(str).str.upper()
        df_opt = df_opt[df_opt["call_put"].isin(["C","CALL"])].copy()
        print(f"Call options rows: {len(df_opt):,}")

    # Join spot price from daily data
    spot_lookup = daily_raw[["symbol","date","close","adj_close"]].copy()
    spot_lookup["date"] = pd.to_datetime(spot_lookup["date"])

    df_opt = df_opt.merge(
        spot_lookup.rename(columns={"date": "trade_date"}),
        on=["symbol","trade_date"], how="left"
    )

    df_opt = df_opt.dropna(subset=["adj_close","strike","expiration"])

    # Derived option features
    df_opt["spot"]          = df_opt["adj_close"]
    df_opt["dte"]           = (df_opt["expiration"] - df_opt["trade_date"]).dt.days
    df_opt["moneyness"]     = (df_opt["strike"] / df_opt["spot"]) - 1.0
    df_opt["log_moneyness"] = np.log(df_opt["strike"] / df_opt["spot"])

    if {"bid","ask"}.issubset(df_opt.columns):
        df_opt["mid"]    = (df_opt["bid"] + df_opt["ask"]) / 2.0
        df_opt["spread"] = (df_opt["ask"] - df_opt["bid"]).clip(lower=0)

    # Assign bucket labels
    df_opt["bucket_label"] = df_opt.apply(
        lambda r: make_bucket_label(r["spot"], r["strike"], r["dte"]), axis=1
    )
    df_calls = df_opt[df_opt["bucket_label"].notna()].copy()

    df_calls.to_parquet(FEAT / "ALL_calls_bucketed.parquet", index=False)
    df_calls.to_csv(FEAT / "ALL_calls_bucketed.csv", index=False)

    print(f"\n Bucketed calls: {df_calls.shape}")
    print("\nBucket distribution:")
    print(df_calls["bucket_label"].value_counts().sort_index())

else:
    df_calls = pd.DataFrame()
    print(" No options data available.")
    print("   Bucket classification skipped. Target labels cannot be constructed from real options.")
    print("   → Upgrade to Alpha Vantage premium key for historical options chains.")

---
## Section 5 — Target Label Construction (Sharpe-Based)
For each `(symbol, trade_date)`, simulates all 9 covered call payoffs at expiration  
and selects the bucket with the highest **realized Sharpe ratio** as the classification target `y`.

In [ ]:
def simulate_covered_call_payoff(spot_entry: float,
                                  spot_exit: float,
                                  strike: float,
                                  premium: float) -> dict:
    """
    Covered call P&L for one option cycle:
      - Stock P&L capped at strike if assigned
      - Seller always keeps full premium

    Returns total_return as fraction of entry price.
    """
    assigned   = spot_exit > strike
    stock_pnl  = (min(spot_exit, strike) - spot_entry) if assigned else (spot_exit - spot_entry)
    total_ret  = (stock_pnl + premium) / spot_entry
    return {
        "total_return": total_ret,
        "assigned"    : assigned,
        "stock_pnl"   : stock_pnl,
    }


def compute_target_labels(df_calls: pd.DataFrame,
                           daily_prices: pd.DataFrame,
                           risk_free_rate: float = RISK_FREE) -> pd.DataFrame:
    """
    For each (symbol, trade_date, bucket_label):
      1. Look up spot price at expiration
      2. Compute realized covered call return per contract
      3. Compute Sharpe proxy across contracts in the bucket
      4. Select best bucket (highest Sharpe) per (symbol, trade_date)

    Returns:
      DataFrame with columns: symbol, trade_date, bucket_label,
                              mean_return, std_return, sharpe, best_bucket
    """
    if df_calls.empty:
        return pd.DataFrame()

    required = {"symbol","trade_date","expiration","strike","spot","bucket_label"}
    missing  = required - set(df_calls.columns)
    if missing:
        raise ValueError(f"Missing columns for label construction: {missing}")

    mid_col = "mid" if "mid" in df_calls.columns else ("ask" if "ask" in df_calls.columns else None)
    if mid_col is None:
        raise ValueError("Options data needs 'bid'/'ask' or 'mid' column to compute premium.")

    # Build fast spot price lookup: (symbol, date) → adj_close
    price_lookup = (
        daily_prices
        .assign(date=pd.to_datetime(daily_prices["date"]))
        .set_index(["symbol","date"])["adj_close"]
        .to_dict()
    )

    records = []
    for (sym, td, bucket), grp in df_calls.groupby(["symbol","trade_date","bucket_label"]):
        for _, row in grp.iterrows():
            spot_exit = price_lookup.get((sym, row["expiration"]), np.nan)
            premium   = row.get(mid_col, np.nan)
            if pd.isna(spot_exit) or pd.isna(premium):
                continue
            pnl = simulate_covered_call_payoff(row["spot"], spot_exit, row["strike"], premium)
            records.append({
                "symbol"      : sym,
                "trade_date"  : td,
                "bucket_label": bucket,
                "total_return": pnl["total_return"],
                "assigned"    : pnl["assigned"],
            })

    if not records:
        print(" No valid payoff records — check options expiration dates against daily price range.")
        return pd.DataFrame()

    df_pnl  = pd.DataFrame(records)
    rf_adj  = risk_free_rate / 252   # per-day adjustment

    summary = (
        df_pnl
        .groupby(["symbol","trade_date","bucket_label"])["total_return"]
        .agg(mean_return="mean", std_return="std")
        .reset_index()
    )
    summary["sharpe"] = (
        (summary["mean_return"] - rf_adj) /
        summary["std_return"].replace(0, np.nan)
    )

    # Assign best_bucket flag per (symbol, trade_date)
    idx_best = summary.groupby(["symbol","trade_date"])["sharpe"].idxmax()
    summary["best_bucket"] = False
    summary.loc[idx_best.dropna(), "best_bucket"] = True

    return summary

print(" compute_target_labels defined")

In [ ]:
# Compute Target Labels 
if not df_calls.empty:
    target_labels = compute_target_labels(df_calls, daily_raw)

    if not target_labels.empty:
        target_labels.to_parquet(FEAT / "ALL_target_labels.parquet", index=False)
        target_labels.to_csv(FEAT / "ALL_target_labels.csv", index=False)

        print(f" Target labels: {target_labels.shape}")
        print("\nBest bucket distribution (classification targets):")
        best = target_labels[target_labels["best_bucket"]]
        print(best["bucket_label"].value_counts().sort_index())
        print(f"\nTotal decision points: {len(best):,}")
    else:
        target_labels = pd.DataFrame()
else:
    target_labels = pd.DataFrame()
    print(" No options data → target labels not computed.")


## Section 6 — Final Dataset Assembly & Export
Joins price features, fundamentals, and target labels into a single modeling-ready DataFrame.

In [ ]:
# Always save features-only dataset 
daily_with_fundamentals["date"] = pd.to_datetime(daily_with_fundamentals["date"])
daily_with_fundamentals.to_parquet(FINAL / "features_only.parquet", index=False)
daily_with_fundamentals.to_csv(FINAL / "features_only.csv", index=False)
print(f" Features-only dataset saved: {daily_with_fundamentals.shape}")

In [ ]:
#  Assemble Full Modeling Dataset (if labels exist) 
if not target_labels.empty:
    # One row per decision point (best bucket only)
    labels_best = (
        target_labels[target_labels["best_bucket"]]
        [["symbol","trade_date","bucket_label","sharpe","mean_return","std_return"]]
        .rename(columns={"bucket_label": "target_bucket", "trade_date": "date"})
        .copy()
    )
    labels_best["date"] = pd.to_datetime(labels_best["date"])

    # Join features onto decision dates
    final_df = labels_best.merge(
        daily_with_fundamentals,
        on=["symbol","date"],
        how="left"
    )

    # Integer class label (0–8) for ML models
    final_df["target_class"] = final_df["target_bucket"].map(BUCKET_TO_INT)

    final_df = final_df.sort_values(["symbol","date"]).reset_index(drop=True)
    final_df.to_parquet(FINAL / "modeling_dataset.parquet", index=False)
    final_df.to_csv(FINAL    / "modeling_dataset.csv",     index=False)

    print(f"Full modeling dataset saved: {final_df.shape}")
    print(f"\nTarget class distribution:")
    print(final_df[["target_bucket","target_class"]].value_counts().sort_index())
    final_df.head(5)

else:
    print(" Target labels not available — only features_only.parquet produced.")
    print("   Provide historical options data and re-run Sections 4–6 to get the labeled dataset.")

In [ ]:
# Final Summary 
print("=" * 65)
print("PREPROCESSING SUMMARY")
print("=" * 65)
print(f"Universe                : {TICKERS}")
print(f"Date Range              : {DATE_START} → {DATE_END}")
print()
print(f"Daily price rows        : {len(daily_raw):,}")
print(f"After price engineering : {len(daily_features):,} rows, {daily_features.shape[1]} cols")
if not quarterly_features.empty:
    print(f"Quarterly fund rows     : {len(quarterly_features):,}")
print(f"Daily + fundamentals    : {daily_with_fundamentals.shape}")
if not df_calls.empty:
    print(f"Bucketed calls          : {df_calls.shape}")
if not target_labels.empty:
    print(f"Target label rows       : {len(target_labels):,}")
    print(f"Labeled decision points : {target_labels['best_bucket'].sum():,}")
print()
print("Output files → data/final/")
print("  features_only.parquet     ← always produced")
if not target_labels.empty:
    print("  modeling_dataset.parquet  ← labeled, ready for ML")
print("=" * 65)
print(" Next step: 03_eda.ipynb  (Week 3) or  03_modeling.ipynb  (Week 5)")